In [1]:
import torch
import torchvision
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
torch.manual_seed(47)


import numpy as np
import math

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


Data

In [2]:
dataset = torchvision.datasets.MNIST(root='./data', download=True, transform=torchvision.transforms.ToTensor())
print(dataset[0][0].shape)

100%|██████████| 9.91M/9.91M [00:02<00:00, 3.59MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 153kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.33MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 1.07MB/s]

torch.Size([1, 28, 28])


Model

In [8]:
class MultilayerPerceptron(nn.Module):
    def __init__(self, input_features, hidden_features, output_features):
        super(MultilayerPerceptron, self).__init__()
        self.flatten = nn.Flatten()
        self.inp = nn.Linear(input_features, hidden_features)
        self.relu = nn.ReLU()
        self.out = nn.Linear(hidden_features, output_features)

    def forward(self, x):
        x = self.flatten(x)
        x = self.inp(x)
        x = self.relu(x)
        x = self.out(x)
        return x

n_features = dataset[0][0].numel()
n_hidden = 256
n_class = 10

Training loop

In [ ]:
model = MultilayerPerceptron(n_features, n_hidden, n_class)

lr = 0.05
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=lr)


dataloader = DataLoader(dataset=dataset, batch_size=512, shuffle=True)

n_epochs = 20
for epoch in range(n_epochs + 1):
    for i, (X, y) in enumerate(dataloader):
        if (device != 'cpu'):
            X, y = X.to(device), y.to(device)

        y_pred = model(X)
        loss = criterion(y_pred, y)

        loss.backward()
        optimizer.step()

        optimizer.zero_grad()

    print(f"Epoch: {epoch}; Loss: {loss:.7f}")

Epoch: 0; Loss: 0.8421970
Epoch: 1; Loss: 0.4026512
Epoch: 2; Loss: 0.3919257
Epoch: 3; Loss: 0.3029035
Epoch: 4; Loss: 0.3110602
Epoch: 5; Loss: 0.3430782
Epoch: 6; Loss: 0.2600367
Epoch: 7; Loss: 0.3879882
Epoch: 8; Loss: 0.2560484
Epoch: 9; Loss: 0.4315068
Epoch: 10; Loss: 0.3290248
Epoch: 11; Loss: 0.2341759
Epoch: 12; Loss: 0.3589956
Epoch: 13; Loss: 0.2473449
Epoch: 14; Loss: 0.3070308
Epoch: 15; Loss: 0.1800467
Epoch: 16; Loss: 0.3740854
Epoch: 17; Loss: 0.2721308
Epoch: 18; Loss: 0.1869407
Epoch: 19; Loss: 0.3164199


Test

In [ ]:
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=torchvision.transforms.ToTensor())
test_loader = DataLoader(dataset=test_dataset, batch_size=1024, shuffle=False)

def evaluate(model, loader):
    total_loss, correct = 0.0, 0
    with torch.no_grad():
        for X, y in loader:
            if (device != 'cpu'):
                X, y = X.to(device), y.to(device)
                
            y_pred = model(X)
            total_loss += criterion(y_pred, y).item()
            correct += (y_pred.argmax(dim=1) == y).sum().item()

    return total_loss / len(loader), correct / len(loader.dataset)

print(evaluate(model, test_loader))

(0.22715932875871658, 0.9352)
